# An independent IRI benchmark for the AKI calls (notebook 94)

**The question.** Our AKI calls cannot be judged inside our own design. Two IR and two control mice share an A2/A4
factor, the screens fail calibration, and notebook 76 could not claim transport loss, its progression or where the
cell-cycle program sits. Which method's AKI calls hold in an independent ischemia–reperfusion experiment with more
mice?

**The benchmark.** Kirita et al. 2020 (PNAS; GSE139107): male C57BL/6J mice, bilateral 18-min IRI, sham and 4 h, 12 h,
2 d, 14 d and 6 weeks after, three mice per group, single-nucleus RNA-seq with author PT labels (S1, S2, S3, injured
PT, failed-repair PT). Mice are the replicates.

**The frozen protocol** is `docs/results/pt-external-iri.md` (commit 9da52b7, amended before any fit in 1599681).
Every call list is read from earlier notebooks and never refit:
- genes: notebook 91's level and location genes, the positional F-test's genes (71), clustering's genes (71);
- sets: the positional F-test (71, 76), clustering (71), position-split ORA (93), notebook 91's level and location
  sets, and the published injury programs (80, without Kirita's own).

**Sections.**
1. Setup and guards.
2. External pseudobulks.
3. External gene statistics: level and location per time point; the assigned-nuclei sensitivity.
4. Our statistics and the call lists.
5. External set statistics.
6. Time-point alignment.
7. Scores, comparisons and verdicts.
8. Threshold-free agreement.
9. Claim checks.
10. Figures.
11. How the protocol was read, summary and run record.

## 1 · Setup

In [ ]:
import argparse
import gzip
import hashlib
import json
import os
import sys
import time
import warnings
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
KIRITA = data_root / 'external' / 'kirita2020_iri'
STATE = data_root / 'external' / 'kidney_physiological_state'
XIONG = data_root / 'external' / 'xiong2023_sex_bias'
T71 = results_root / 'pt_pathways_package_pseudospace'
T76 = results_root / 'pt_package_aki' / 'tables'
T80 = results_root / 'pt_known_injury_programs' / 'tables'
T91 = results_root / 'pt_positional_decomposition' / 'tables'
T93 = results_root / 'pt_pathway_networks' / 'tables'
output = results_root / 'pt_external_iri'
tables, figures, cache = output / 'tables', output / 'figures', output / 'stage_cache'
for folder in (tables, figures, cache):
    folder.mkdir(parents=True, exist_ok=True)

NOTEBOOK_LOGIC_VERSION = '94.external_iri.1'   # Bump when a cached calculation changes.
SEED = 94
FROZEN_SHA = {
    'GSE139107_MouseIRI.metadata.txt.gz': 'ac607fa70c7ee0c6b269009893cb0ca06b463b3c846db613d7dec6a2b5ee9a11',
    'GSE139107_MouseIRI_control.dge.txt.gz': '626a4a354bd0263cbf68f91cb2de1304e629a7bfa9664aa34cbb899fb3427628',
    'GSE139107_MouseIRI_4hours.dge.txt.gz': '21ef8eae053a069f2e31a0cdf870aeeabf4a90d83b33d37ce06545bb39486321',
    'GSE139107_MouseIRI_12hours.dge.txt.gz': '5bb3a2bae56c90fb854e655d13c1ce032f0cdf5c14ab97a06521ef863769f732',
    'GSE139107_MouseIRI_2days.dge.txt.gz': '652b258719afea520a5c44b13f7bb4e4a964b549242e5ec921498c20c23dbdd6',
    'GSE139107_MouseIRI_14days.dge.txt.gz': 'a0a249ddb5b646f882ef4e9955821c4e71e40b23ac9df2b8999780889151d6bc',
    'GSE139107_MouseIRI_6weeks.dge.txt.gz': '7b80febcc52a936ffa42e8930debed2d793fcfb4d2896de415f2b45a6792c3af',
}
GROUP_FILE = {'Control': 'control', '4hours': '4hours', '12hours': '12hours', '2days': '2days', '14days': '14days',
              '6weeks': '6weeks'}
TIMES = ['4hours', '12hours', '2days', '14days', '6weeks']
ACUTE = ['4hours', '12hours', '2days']
T_LABEL = {'4hours': '4 h', '12hours': '12 h', '2days': '2 d', '14days': '14 d', '6weeks': '6 wk'}
PT_TYPES = ['PTS1', 'PTS2', 'PTS3', 'NewPT1', 'NewPT2']
SEGMENTS = ['PTS1', 'PTS2', 'PTS3']
FROZEN_TAB = {  # PT nuclei per group (protocol G2)
    'Control': [4750, 4106, 1937, 153, 54], '4hours': [1373, 2293, 460, 2522, 264],
    '12hours': [1273, 1841, 361, 2783, 355], '2days': [4062, 3580, 516, 806, 529],
    '14days': [1883, 2481, 1057, 713, 2062], '6weeks': [2897, 2812, 1511, 189, 552]}
FROZEN_CALLS = {'F71': 103, 'C71': 459, 'N93': 134, 'F76': 106, 'level genes': 3159, 'location genes': 362,
                'level sets': 538, 'location sets': 236}
ALPHA = .05
MIN_CPM, MIN_MICE = 1., 3                     # universe E: whole-PT CPM >= 1 in >= 3 mice
SET_SIZES = (10, 300)
MIN_PROGRAM = 5
CAMERA_COR = .01
N_MARKERS, MARKER_CPM = 50, 10.
N_NULL = 1000
LATE = 1.8                                    # 91's LATE (amended protocol)
MIN_N_COMPARE, MIN_N_REPLICATE, MARGIN = 20, 10, .10
MT_PREFIX = 'mt-'
TOD_LFC, TOD_P, FAST_LFC, FAST_CPM = .5, .01, 1., 10.   # 91's external-program constants

REQUIRED = [*(KIRITA / f for f in FROZEN_SHA),
            T71 / 'tables' / 'universe_aki_vs_control.csv', T71 / 'tables' / 'gene_statistics_aki_vs_control.csv',
            T71 / 'tables' / 'gene_sets_aki_vs_control.json',
            T71 / 'tables' / 'pathway_tests_aki_vs_control_cameraPR_species.csv',
            T76 / 'universe.csv', T76 / 'gene_sets.json', T76 / 'pathway_tests_condition.csv', T76 / 'programs.csv',
            T76 / 'positive_control_markers.csv', T80 / 'programs_members.csv', T80 / 'programs.csv',
            T91 / 'i_genes.csv.gz', T91 / 'i5_library_sets.csv', T91 / 'i_induced_genes.csv', T91 / 'i_lost_genes.csv',
            T93 / 'calls_aki_vs_control.csv', T93 / 'row_n_aki_vs_control.csv',
            STATE / 'GSE267280_RNA_Seq_counts.txt.gz', STATE / 'GSM9744085_CTRL_ZT4_01_features.tsv.gz',
            *[XIONG / f'PT-{s}_fWTvsmWT.csv' for s in ('S1', 'S2', 'S3')]]
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (download GSE139107; run notebooks 71, 76, 80, 91 and 93 first)')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.stats import norm, spearmanr
from IPython.display import display

MM = 1 / 25.4
INK, MUTED, LIGHT = '#0b0b0b', '#52514e', '#d9d8d4'
LIST_COLOR = {'G-L91': '#2a78d6', 'G-M91': '#9C93D3', 'G-F71': '#1b8a5a', 'G-C71': '#eda100',
              'S-F71': '#1b8a5a', 'S-F76': '#5fb88a', 'S-C71': '#eda100', 'S-N93': '#d55e00',
              'S-L91': '#2a78d6', 'S-M91': '#9C93D3', 'S-P80': '#52514e'}
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5,
    'axes.titleweight': 'bold', 'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 6, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': .6, 'xtick.major.width': .6, 'ytick.major.width': .6, 'xtick.major.size': 2.5,
    'ytick.major.size': 2.5, 'lines.linewidth': 1.0, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_colwidth', 70, 'display.max_rows', 80)


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')


print('Results folder:', output)

### 1.1 · Guards (not cached)

- G1: the seven external files match the frozen sha256.
- G2: the metadata reproduces the frozen tabulation (nuclei, libraries, mice, PT nuclei per group and type).
- G3: every metadata barcode is a column of exactly one count matrix (headers here; whole-number counts are checked
  while the matrices are read).
- G4 runs in section 4, where the call lists are read.

In [ ]:
guard_log = []
for name, sha in FROZEN_SHA.items():                                                                         # G1
    assert hashlib.sha256((KIRITA / name).read_bytes()).hexdigest() == sha, f'G1: {name} changed'
guard_log.append('G1: the seven GSE139107 files match the frozen sha256')

meta = pd.read_csv(KIRITA / 'GSE139107_MouseIRI.metadata.txt.gz', sep='\t', index_col=0)                    # G2
meta['Replicates'] = meta.Replicates.astype(str)
meta['library'] = meta.Group + '|' + meta.Replicates
meta['mouse'] = meta.Group + '|' + meta.Replicates.str.split('_').str[0]
assert len(meta) == 126578 and meta.index.is_unique, 'G2: nucleus count'
assert meta.library.nunique() == 24 and meta.mouse.nunique() == 18, 'G2: libraries or mice'
assert meta.groupby('Group').mouse.nunique().eq(3).all(), 'G2: three mice per group'
PT = meta[meta.celltype.isin(PT_TYPES)].copy()
tab = PT.groupby(['Group', 'celltype']).size().unstack().reindex(columns=PT_TYPES).fillna(0).astype(int)
assert all(tab.loc[g].tolist() == v for g, v in FROZEN_TAB.items()), 'G2: PT nuclei per group'
guard_log.append('G2: 126,578 nuclei, 24 libraries, 18 mice (3 per group), PT nuclei per group and type as frozen')

seen = {}                                                                                                     # G3
for group, stem in GROUP_FILE.items():
    with gzip.open(KIRITA / f'GSE139107_MouseIRI_{stem}.dge.txt.gz', 'rt') as handle:
        header = handle.readline().rstrip('\n').split('\t')
    for b in header:
        assert b not in seen, f'G3: {b} in two matrices'
        seen[b] = group
assert all(seen.get(b) == g for b, g in zip(meta.index, meta.Group)), 'G3: a metadata barcode is not in its matrix'
guard_log.append('G3: every metadata barcode is a column of exactly one matrix, the one of its group')
del seen
print('\n'.join('· ' + g for g in guard_log))

## 2 · External pseudobulks

PT nuclei summed per mouse × author type (libraries of one mouse pooled). Library = summed counts over every gene.
Kept as a sparse matrix for the assigned-nuclei sensitivity: the sham mice's segment nuclei and every injured
(NewPT1, NewPT2) nucleus.

In [ ]:
from pseudospace.external_replication import (assign_by_centroid, dge_group_sums, log_cpm, matched_null,
                                              null_summary, replicated, segment_markers)
from pseudospace.stage_cache import cached_payload

PT['column'] = PT.mouse + '|' + PT.celltype
KEEP = PT.index[(PT.celltype.isin(['NewPT1', 'NewPT2'])) | (PT.Group.eq('Control') & PT.celltype.isin(SEGMENTS))]
KEEP_SET = set(KEEP)


def build_pseudobulks():
    frames, kept, kept_barcodes = [], [], []
    for group, stem in GROUP_FILE.items():
        sub = PT[PT.Group.eq(group)]
        keep = [b for b in sub.index if b in KEEP_SET]
        t0 = time.perf_counter()
        genes, sums, matrix = dge_group_sums(KIRITA / f'GSE139107_MouseIRI_{stem}.dge.txt.gz',
                                             dict(zip(sub.index, sub.column)), keep=keep)
        frames.append(sums)
        kept.append((genes, matrix.astype(np.int32)))
        kept_barcodes.extend(keep)
        print(f'  {group}: {len(genes):,} genes, {sub.shape[0]:,} PT nuclei, {time.perf_counter() - t0:.0f} s')
    table = pd.concat(frames, axis=1).fillna(0)
    genes = table.index.astype(str).tolist()
    position = {g: i for i, g in enumerate(genes)}
    blocks = []
    for g_list, m in kept:                                    # align each file's genes to the union
        cols = np.array([position[g] for g in g_list])
        m = m.tocoo()
        blocks.append(sparse.csr_matrix((m.data, (m.row, cols[m.col])), shape=(m.shape[0], len(genes)), dtype=np.int32))
    keptm = sparse.vstack(blocks).tocsr()
    return {'genes': np.array(genes), 'columns': np.array(table.columns.astype(str)), 'sums': table.to_numpy(np.int64),
            'kept_barcodes': np.array(kept_barcodes), 'kept_data': keptm.data, 'kept_indices': keptm.indices,
            'kept_indptr': keptm.indptr, 'kept_shape': np.array(keptm.shape)}


t0 = time.perf_counter()
PB = cached_payload('pseudobulks', build_pseudobulks, root=cache,
                    params={'logic': NOTEBOOK_LOGIC_VERSION, 'types': PT_TYPES},
                    inputs={k: v for k, v in FROZEN_SHA.items()}, code=dge_group_sums)
KG = PB['genes'].astype(str).tolist()
SUMS = pd.DataFrame(PB['sums'], index=KG, columns=PB['columns'].astype(str))
KEPT = sparse.csr_matrix((PB['kept_data'], PB['kept_indices'], PB['kept_indptr']), shape=tuple(PB['kept_shape']))
KEPT_BARCODES = PB['kept_barcodes'].astype(str)
assert set(KEPT_BARCODES) == set(KEEP)
LIB = SUMS.sum(axis=0)
MICE = sorted(PT.mouse.unique(), key=lambda m: (list(GROUP_FILE).index(m.split('|')[0]), m))
MOUSE_GROUP = {m: m.split('|')[0] for m in MICE}
WHOLE = pd.DataFrame({m: SUMS[[f'{m}|{t}' for t in PT_TYPES if f'{m}|{t}' in SUMS]].sum(axis=1) for m in MICE})
NUCLEI = PT.groupby(['mouse', 'celltype']).size().unstack().reindex(index=MICE, columns=PT_TYPES).fillna(0).astype(int)
NUCLEI['libraries'] = PT.groupby('mouse').library.nunique().reindex(MICE)
NUCLEI['whole-PT counts'] = WHOLE.sum(axis=0).reindex(MICE).astype(int)
NUCLEI.to_csv(tables / 'nuclei.csv')
SUMS.to_csv(tables / 'pseudobulk_counts.csv.gz')            # mouse x author type sums, for later notebooks
display(NUCLEI)
print(f'{len(KG):,} genes in the union of the six matrices · pseudobulks {time.perf_counter() - t0:.0f} s')

## 3 · External gene statistics

**Universe E:** genes of our AKI universes (71, U76, 91's shared, induced and lost genes) with an exact symbol in the
matrices, whole-PT CPM ≥ 1 in ≥ 3 of the 18 mice, mt genes out.

**Level** per time point t: limma-trend (robust) on whole-PT log2 CPM, 3 IRI-t mice against 3 sham mice; the
coefficient centred by its median over E; BH within t. **Location:** the same on d = log2 CPM(S3) − log2 CPM(S1) per
mouse (and S3 − S2, S2 − S1), identity-retaining nuclei only.

In [ ]:
from pseudospace.species_level import limma_trend_species

U71 = pd.read_csv(T71 / 'tables' / 'universe_aki_vs_control.csv').gene.astype(str).tolist()
U76 = pd.read_csv(T76 / 'universe.csv').gene.astype(str).tolist()
G91 = pd.read_csv(T91 / 'i_genes.csv.gz').set_index('gene')
G91.index = G91.index.astype(str)
INDUCED = pd.read_csv(T91 / 'i_induced_genes.csv').gene.astype(str).tolist()
LOST = pd.read_csv(T91 / 'i_lost_genes.csv').gene.astype(str).tolist()
candidates = set(U71) | set(U76) | set(G91.index) | set(INDUCED) | set(LOST)
whole_cpm = WHOLE / WHOLE.sum(axis=0) * 1e6
expressed = (whole_cpm >= MIN_CPM).sum(axis=1) >= MIN_MICE
E = sorted(g for g in candidates if g in SUMS.index and expressed[g] and not g.startswith(MT_PREFIX))
UNIVERSE = pd.DataFrame({'gene': E, 'in 71': np.isin(E, U71), 'in U76': np.isin(E, U76), 'in 91 shared': np.isin(E, G91.index),
                         '91 induced': np.isin(E, INDUCED), '91 lost': np.isin(E, LOST)})
UNIVERSE.to_csv(tables / 'universe.csv', index=False)
print(f'E: {len(E):,} genes of {len(candidates):,} candidates '
      f'({sum(g in SUMS.index for g in candidates):,} with a symbol in the matrices)')


def whole_values(mice):
    counts = WHOLE.loc[E, mice].to_numpy(float)
    return log_cpm(counts, WHOLE[mice].sum(axis=0).to_numpy(float)).T


def segment_values(mice, segment, table=None, libraries=None):
    table = SUMS if table is None else table
    libraries = LIB if libraries is None else libraries
    cols = [f'{m}|{segment}' for m in mice]
    return log_cpm(table.loc[E, cols].to_numpy(float), libraries[cols].to_numpy(float)).T


def limma_frame(values, case, prefix):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        fit = limma_trend_species(values, case, genes=E, trend=True, robust=True)
    return pd.DataFrame({f'{prefix} coefficient': fit.centred, f'{prefix} t': fit.t_centred, f'{prefix} p': fit.p,
                         f'{prefix} q': fit.q}, index=E)


SHAM = [m for m in MICE if MOUSE_GROUP[m] == 'Control']
CONTRASTS = {'location': ('PTS3', 'PTS1'), 'S3-S2': ('PTS3', 'PTS2'), 'S2-S1': ('PTS2', 'PTS1')}


def external_genes(table=None, libraries=None, tag=''):
    out = {}
    for t in TIMES:
        mice = SHAM + [m for m in MICE if MOUSE_GROUP[m] == t]
        case = np.array([MOUSE_GROUP[m] == t for m in mice])
        parts = [] if tag else [limma_frame(whole_values(mice), case, 'level')]
        for name, (a, b) in CONTRASTS.items():
            if tag and name != 'location':
                continue
            d = segment_values(mice, a, table, libraries) - segment_values(mice, b, table, libraries)
            parts.append(limma_frame(d, case, name + tag))
        out[t] = pd.concat(parts, axis=1)
    return pd.concat(out, axis=1)


t0 = time.perf_counter()
EXT = external_genes()
print(f'External gene statistics {time.perf_counter() - t0:.0f} s')
summary = pd.DataFrame({T_LABEL[t]: {f'{k} q ≤ 0.05': int((EXT[t][f'{k} q'] <= ALPHA).sum()) for k in ('level', *CONTRASTS)}
                        for t in TIMES})
display(summary)

### 3.1 · Location sensitivity: injured nuclei assigned to a segment

Markers: the 50 genes in E with the largest one-vs-rest log2 CPM difference per segment in the sham mice (CPM ≥ 10
there). Each injured nucleus goes to the sham segment centroid of highest Spearman r (log1p CP10k over the markers).
Accuracy: leave one sham mouse out.

In [ ]:
gene_pos = pd.Series(np.arange(len(KG)), index=KG)
kept_meta = PT.loc[KEPT_BARCODES]
nucleus_total = np.asarray(KEPT.sum(axis=1)).ravel()


def markers_from(mice):
    counts = pd.DataFrame({s: SUMS[[f'{m}|{s}' for m in mice]].sum(axis=1) for s in SEGMENTS}).loc[E]
    libs = np.array([LIB[[f'{m}|{s}' for m in mice]].sum() for s in SEGMENTS])
    return segment_markers(pd.DataFrame(log_cpm(counts.to_numpy(float), libs), index=E, columns=SEGMENTS),
                           counts / libs * 1e6, n=N_MARKERS, min_cpm=MARKER_CPM)


def profiles(rows, genes):
    block = KEPT[rows][:, gene_pos[genes].to_numpy()].toarray().astype(float)
    return np.log1p(block / nucleus_total[rows, None] * 1e4)


def centroids_from(mice, genes):
    rows_by = [np.flatnonzero(kept_meta.mouse.isin(mice).to_numpy() & kept_meta.celltype.eq(s).to_numpy()) for s in SEGMENTS]
    return np.vstack([profiles(r, genes).mean(axis=0) for r in rows_by])


ACC = []
for held in SHAM:
    rest = [m for m in SHAM if m != held]
    mk = markers_from(rest)
    genes = [g for s in SEGMENTS for g in mk[s]]
    rows = np.flatnonzero(kept_meta.mouse.eq(held).to_numpy() & kept_meta.celltype.isin(SEGMENTS).to_numpy())
    choice, _ = assign_by_centroid(profiles(rows, genes), centroids_from(rest, genes))
    truth = kept_meta.celltype.to_numpy()[rows]
    pred = np.array(SEGMENTS + ['none'])[choice]
    ACC.append({'held-out sham mouse': held, 'nuclei': len(rows), 'accuracy': float((pred == truth).mean()),
                **{f'recall {s}': float((pred[truth == s] == s).mean()) for s in SEGMENTS}})
ACC = pd.DataFrame(ACC)
ACC.to_csv(tables / 'assignment_accuracy.csv', index=False)
display(ACC.round(3))

MARKERS = markers_from(SHAM)
MARKER_GENES = [g for s in SEGMENTS for g in MARKERS[s]]
injured_rows = np.flatnonzero(kept_meta.celltype.isin(['NewPT1', 'NewPT2']).to_numpy())
choice, _ = assign_by_centroid(profiles(injured_rows, MARKER_GENES), centroids_from(SHAM, MARKER_GENES))
assigned = pd.Series(np.array(SEGMENTS + ['none'])[choice], index=KEPT_BARCODES[injured_rows])
ASSIGNED_SUMS = SUMS.copy()
for (mouse, segment), barcodes in assigned.groupby([kept_meta.loc[assigned.index].mouse, assigned]).groups.items():
    if segment == 'none':
        continue
    rows = np.flatnonzero(np.isin(KEPT_BARCODES, list(barcodes)))
    ASSIGNED_SUMS[f'{mouse}|{segment}'] += np.asarray(KEPT[rows].sum(axis=0)).ravel()
ASSIGNED_LIB = ASSIGNED_SUMS.sum(axis=0)
ASSIGNED_COUNTS = pd.crosstab(kept_meta.loc[assigned.index].mouse.map(MOUSE_GROUP), assigned).reindex(list(GROUP_FILE))
display(ASSIGNED_COUNTS)
EXT_A = external_genes(ASSIGNED_SUMS, ASSIGNED_LIB, tag=' (assigned)')
EXT = pd.concat([EXT, EXT_A], axis=1).sort_index(axis=1, level=0, sort_remaining=False)
EXT = EXT.reindex(columns=pd.MultiIndex.from_tuples([(t, c) for t in TIMES for c in EXT[t].columns]))
EXT.to_csv(tables / 'external_genes.csv.gz')

## 4 · Our statistics and the call lists

Internal signs come from notebook 91 for every list: genes' level sign = sign(L), location sign = sign(Δ late − L);
sets' level sign = sign(L), location sign = sign of mean P over the bins centred ≥ 1.8 (91's LATE).

**Guard G4:** each list's call count equals the recorded one.

In [ ]:
S91 = pd.read_csv(T91 / 'i5_library_sets.csv').set_index('set')
late_cols = [c for c in S91.columns if c.startswith('P ') and float(c.split()[1]) >= LATE]
assert [float(c.split()[1]) for c in late_cols] == [1.85, 1.95, 2.05, 2.15, 2.25, 2.35, 2.45]
SET_SIGN = {'level': np.sign(S91.L), 'location': np.sign(S91[late_cols].mean(axis=1))}
GENE_SIGN = {'level': np.sign(G91.L), 'location': np.sign(G91['Δ late'] - G91.L)}

stat71 = pd.read_csv(T71 / 'tables' / 'gene_statistics_aki_vs_control.csv', header=[0, 1], index_col=0)['condition']
stat71.index = stat71.index.astype(str)
cam71 = pd.read_csv(T71 / 'tables' / 'pathway_tests_aki_vs_control_cameraPR_species.csv').set_index('pathway_id')
cam76 = pd.read_csv(T76 / 'pathway_tests_condition.csv').set_index('pathway_id')
calls93 = pd.read_csv(T93 / 'calls_aki_vs_control.csv')
rown = pd.read_csv(T93 / 'row_n_aki_vs_control.csv', header=[0, 1], index_col=0)['condition']
SETS71 = json.loads((T71 / 'tables' / 'gene_sets_aki_vs_control.json').read_text())
SETS76 = json.loads((T76 / 'gene_sets.json').read_text())


def stored(folder, stage):
    files = sorted((folder / 'stage_cache').glob(f'{stage}__*.npz'),
                   key=lambda p: json.loads(Path(str(p) + '.key.json').read_text())['written_utc'])
    if not files:
        raise FileNotFoundError(f'{folder.name}: no cached {stage}')
    with np.load(files[-1], allow_pickle=False) as payload:
        return {k: payload[k] for k in payload.files}


clus = stored(T71, 'method_b_aki_vs_control_species__null_default')
clus_genes = pd.DataFrame({c: clus[f'genes|{c}'] for c in clus['genes|__columns'].astype(str)})
clus_genes['gene'] = clus_genes.gene.astype(str)
clus_genes = clus_genes.set_index('gene')
clus_sel = clus_genes[clus_genes.selected.astype(bool) & clus_genes.direction.isin(['up', 'down'])]

usable71 = stat71.converged.astype(bool) & ~stat71.separated.astype(bool) & stat71.q.notna()
LISTS = {
    'G-L91': ('gene', G91.index[G91['level change'].astype(bool)].tolist(), None, list(G91.index)),
    'G-M91': ('gene', G91.index[G91['location change'].astype(bool)].tolist(), None, list(G91.index)),
    'G-F71': ('gene', stat71.index[usable71 & (stat71.q <= ALPHA)].tolist(), None, stat71.index[usable71].tolist()),
    'G-C71': ('gene', clus_sel.index.tolist(), pd.Series(np.where(clus_sel.direction.eq('up'), 1., -1.), index=clus_sel.index),
              clus_genes.index.tolist()),
    'S-F71': ('set', cam71.index[cam71.call.astype(bool)].tolist(), None, cam71.index.tolist()),
    'S-F76': ('set', cam76.index[cam76.call.astype(bool)].tolist(), None, cam76.index.tolist()),
    'S-C71': ('set', calls93.pathway_id[calls93.method.eq('clustering')].tolist(), None, list(SETS71)),
    'S-N93': ('set', calls93.pathway_id[calls93.method.eq('position-split ORA')].tolist(), None, rown.index.astype(str).tolist()),
    'S-L91': ('set', S91.index[S91['level set'].astype(bool)].tolist(), None, S91.index.tolist()),
    'S-M91': ('set', S91.index[S91['location set'].astype(bool)].tolist(), None, S91.index.tolist()),
}
recorded = {'F71': len(LISTS['S-F71'][1]), 'C71': len(LISTS['S-C71'][1]), 'N93': len(LISTS['S-N93'][1]),
            'F76': len(LISTS['S-F76'][1]), 'level genes': len(LISTS['G-L91'][1]), 'location genes': len(LISTS['G-M91'][1]),
            'level sets': len(LISTS['S-L91'][1]), 'location sets': len(LISTS['S-M91'][1])}
assert recorded == FROZEN_CALLS, f'G4: {recorded}'
guard_log.append('G4: the read-only call lists have their recorded sizes (F 103, clustering 459, ORA 134, F76 106, '
                 '91: 3,159 / 362 genes, 538 / 236 sets)')
print('· ' + guard_log[-1])

# programs: 80's published injury programs (Kirita's own excluded), the claim-check programs
P80 = pd.read_csv(T80 / 'programs.csv').set_index('program_id')
M80 = pd.read_csv(T80 / 'programs_members.csv')
PROGRAMS, PROGRAM_SIGN = {}, {}
for pid, block in M80.groupby('program_id', sort=False):
    if pid.startswith('K-'):
        continue
    PROGRAMS[f'80 · {pid}'] = block.gene.astype(str).tolist()
    PROGRAM_SIGN[f'80 · {pid}'] = 1. if P80.loc[pid, 'direction'] == 'up in IR' else -1.
assert set(P80.direction[~P80.index.str.startswith('K-')]) == {'up in IR', 'down in IR'}
S_P80 = list(PROGRAMS)
SETS76_U = set(pd.read_csv(T76 / 'universe.csv').gene.astype(str))
PROGRAM_C = sorted((set(SETS76['MSigDB_Hallmark_2020::G2-M Checkpoint']) | set(SETS76['MSigDB_Hallmark_2020::E2F Targets'])) & SETS76_U)
transporters = pd.read_csv(T76 / 'programs_transporters.csv')
PROGRAM_TPT = sorted(transporters.mouse[transporters.in_T_PT.astype(bool)].astype(str))
assert len(PROGRAM_C) == 297 and len(PROGRAM_TPT) == 93, "76's programs C and T_PT"   # 76's programs.csv truncates members
markers76 = pd.read_csv(T76 / 'positive_control_markers.csv', index_col=0).index.astype(str).tolist()
assert len(markers76) == 18
features = pd.read_csv(STATE / 'GSM9744085_CTRL_ZT4_01_features.tsv.gz', sep='\t', header=None, names=['id', 'symbol', 'type'])
ens_to_symbol = dict(zip(features.id, features.symbol))
fast = pd.read_csv(STATE / 'GSE267280_RNA_Seq_counts.txt.gz', sep='\t')
fast['symbol'] = fast.Gene_id.str.split('.').str[0].map(ens_to_symbol)
fast = fast.dropna(subset=['symbol']).groupby('symbol')[['Fed_Kidney', 'Fasting_Kidney']].sum()
fcpm = fast / fast.sum() * 1e6
FAST = pd.DataFrame({'lfc': np.log2(fcpm.Fasting_Kidney + 1) - np.log2(fcpm.Fed_Kidney + 1), 'cpm': fcpm.mean(axis=1)})
xi = pd.concat([pd.read_csv(XIONG / f'PT-{s}_fWTvsmWT.csv').assign(segment=s) for s in ('S1', 'S2', 'S3')])
sig = xi[xi.significant.astype(str).str.upper().eq('TRUE')]
CLAIM_PROGRAMS = {
    'K1 · injury markers (76)': (markers76, 'level', 1.),
    'K2 · cell-cycle program C (76)': (PROGRAM_C, 'level', 1.),
    'K3 · transport program T_PT (76)': (PROGRAM_TPT, 'level', -1.),
    'K4 · T_PT, progressive (76)': (PROGRAM_TPT, 'location', -1.),
    'K7 · male-biased PT (Xiong 2023)': (sorted(set(sig.symbol[sig.Direction.eq('DOWN')])), 'level', -1.),
    'K8 · fasting up (GSE267280)': (FAST.index[(FAST.lfc >= FAST_LFC) & (FAST.cpm >= FAST_CPM)].tolist(), 'level', 1.),
    'K8 · fasting down (GSE267280)': (FAST.index[(FAST.lfc <= -FAST_LFC) & (FAST.cpm >= FAST_CPM)].tolist(), 'level', -1.),
    'K9 · induced (91)': (INDUCED, 'level', 1.),
    'K9 · lost (91)': (LOST, 'level', -1.),
}
K5 = ['KEGG_2019_Mouse::PPAR signaling pathway', 'KEGG_2019_Mouse::Peroxisome',
      'Reactome_2022::Peroxisomal Lipid Metabolism R-HSA-390918', 'Reactome_2022::Cholesterol Biosynthesis R-HSA-191273',
      'Reactome_2022::Plasma Lipoprotein Assembly R-HSA-8963898', 'MSigDB_Hallmark_2020::Fatty Acid Metabolism']
K6 = ['MSigDB_Hallmark_2020::' + n for n in ('G2-M Checkpoint', 'E2F Targets', 'TNF-alpha Signaling via NF-kB',
                                            'p53 Pathway', 'Apoptosis', 'Inflammatory Response', 'Hypoxia')]
print({k: len(v[0]) for k, v in CLAIM_PROGRAMS.items()})

## 5 · External set statistics

cameraPR (inter-gene correlation 0.01) of each library set, on its members in E (10–300), against the external level
t, location t and |location t| (and the assigned-nuclei location), BH within t. Programs (80's, the claim checks) are
tested in their own family on their members in E (≥ 5).

In [ ]:
from pseudospace.pathway_pipelines import camera_pr

E_SET = set(E)
LIB_SETS = {}
for source in (SETS71, SETS76, {s: [] for s in S91.index}):
    for k, v in source.items():
        LIB_SETS.setdefault(k, set()).update(map(str, v))
LIB_SETS = {k: sorted(v & E_SET) for k, v in LIB_SETS.items()}
LIB_SETS = {k: v for k, v in LIB_SETS.items() if SET_SIZES[0] <= len(v) <= SET_SIZES[1]}
PROG_SETS = {k: sorted(set(v) & E_SET) for k, v in {**PROGRAMS, **{k: v[0] for k, v in CLAIM_PROGRAMS.items()}}.items()}
PROG_SETS = {k: v for k, v in PROG_SETS.items() if len(v) >= MIN_PROGRAM}
print(f'{len(LIB_SETS):,} library sets with 10–300 members in E · {len(PROG_SETS)} programs with ≥ 5')
VECTORS = {'level': ('level t', False), 'location': ('location t', False), '|location|': ('location t', True),
           'location (assigned)': ('location (assigned) t', False), '|location (assigned)|': ('location (assigned) t', True)}


def camera_table(sets):
    rows = []
    for t in TIMES:
        for name, (column, absolute) in VECTORS.items():
            v = EXT[t][column]
            r = camera_pr(v.abs() if absolute else v, sets, inter_gene_cor=CAMERA_COR).set_index('pathway_id')
            sign = np.where(r.direction.eq('Up'), 1., -1.)
            rows.append(pd.DataFrame({'time': t, 'statistic': name, 'n in E': r.n, 'direction': r.direction,
                                      'z': sign * norm.isf(r.p / 2), 'p': r.p, 'fdr': r.fdr}, index=r.index))
    return pd.concat(rows).rename_axis('set').reset_index()


t0 = time.perf_counter()
CAM = camera_table(LIB_SETS)
CAMP = camera_table(PROG_SETS)
pd.concat([CAM.assign(family='library'), CAMP.assign(family='program')]).to_csv(tables / 'external_sets.csv.gz', index=False)
print(f'External set statistics {time.perf_counter() - t0:.0f} s')
display(CAM[CAM.fdr <= ALPHA].groupby(['statistic', 'time']).size().unstack().reindex(columns=TIMES).rename(columns=T_LABEL))

## 6 · Time-point alignment

Spearman ρ over E ∩ 91's genes between our L and the external level coefficient; t* is the time point of largest ρ.
The same for location (our Δ late − L against the external S3 − S1 coefficient).

In [ ]:
common = [g for g in E if g in G91.index]
ALIGN = pd.DataFrame({T_LABEL[t]: {
    'level ρ': spearmanr(G91.loc[common, 'L'], EXT[t].loc[common, 'level coefficient'])[0],
    'location ρ': spearmanr(G91.loc[common, 'Δ late'] - G91.loc[common, 'L'], EXT[t].loc[common, 'location coefficient'])[0],
    'location (assigned) ρ': spearmanr(G91.loc[common, 'Δ late'] - G91.loc[common, 'L'],
                                       EXT[t].loc[common, 'location (assigned) coefficient'])[0]} for t in TIMES}).T
ALIGN['genes'] = len(common)
T_STAR = TIMES[int(np.argmax(ALIGN['level ρ'].to_numpy()))]
ALIGN.to_csv(tables / 'alignment.csv')
display(ALIGN.round(3))
print('t* =', T_LABEL[T_STAR])

## 7 · Scores

Replicated: external q (genes) or camera FDR (sets) ≤ 0.05 in the item's internal direction (level, location), or
in either direction (unsigned location: genes |location| q; sets the |location t| camera Up). Null: 1,000 random lists
of the same size from the list's pool, matched on expression decile (genes) or size tercile (sets).

In [ ]:
expr91 = G91['log2 CP10k (controls)'].rank(pct=True)
mean_count = pd.Series(np.asarray(clus_genes['expression_per_10k'], float), index=clus_genes.index).rank(pct=True)
EXPR_PCT = expr91.combine_first(mean_count)
EXPR_DECILE = np.minimum((EXPR_PCT * 10).astype(int), 9)
AXES = {'level': 'level', 'location': 'location', 'unsigned location': 'location',
        'location (assigned)': 'location (assigned)', 'unsigned location (assigned)': 'location (assigned)'}
PRIMARY = {'G-L91': 'level', 'G-M91': 'location', 'G-F71': 'unsigned location', 'G-C71': 'level',
           'S-F71': 'unsigned location', 'S-F76': 'unsigned location', 'S-C71': 'level', 'S-N93': 'location',
           'S-L91': 'level', 'S-M91': 'location', 'S-P80': 'level'}


def gene_hits(t, axis, signs):
    stat = AXES[axis]
    q = EXT[t][f'{stat} q']
    if axis.startswith('unsigned'):
        return q <= ALPHA
    return pd.Series(replicated(EXT[t][f'{stat} coefficient'], q, signs.reindex(E)), index=E)


def set_hits(t, axis, signs, cam):
    stat = {'level': 'level', 'location': 'location', 'unsigned location': '|location|',
            'location (assigned)': 'location (assigned)', 'unsigned location (assigned)': '|location (assigned)|'}[axis]
    block = cam[cam.time.eq(t) & cam.statistic.eq(stat)].set_index('set')
    if axis.startswith('unsigned'):
        return block.direction.eq('Up') & (block.fdr <= ALPHA)
    return pd.Series(replicated(block.z, block.fdr, signs.reindex(block.index)), index=block.index)


rows = []
for name, (kind, items, own_sign, universe) in {**LISTS, 'S-P80': ('set', S_P80, pd.Series(PROGRAM_SIGN), None)}.items():
    for axis in AXES:
        signed = not axis.startswith('unsigned')
        if name == 'S-P80' and axis != 'level':
            continue
        for t in TIMES:
            if kind == 'gene':
                # signs: 91's for the pool; on the level axis a list's own sign replaces 91's for its members
                base = GENE_SIGN['level' if axis == 'level' else 'location'] if signed else pd.Series(1., index=E)
                hits = gene_hits(t, axis, base)
                pool = [g for g in universe if g in hits.index and g in EXPR_DECILE.index and base.get(g, 0) != 0]
                if own_sign is not None and axis == 'level':
                    member_sign = own_sign
                    member_hits = gene_hits(t, axis, own_sign)
                else:
                    member_sign, member_hits = base, hits
                members = [g for g in items if g in hits.index and g in EXPR_DECILE.index and member_sign.get(g, 0) != 0]
                k = int(member_hits.reindex(members).fillna(False).sum())
                strata_pool, strata_list = EXPR_DECILE.reindex(pool).to_numpy(), EXPR_DECILE.reindex(members).to_numpy()
                null = matched_null(hits.reindex(pool).fillna(False).to_numpy(), strata_pool, strata_list, n=N_NULL, seed=SEED)
            else:
                cam = CAMP if name == 'S-P80' else CAM
                signs = SET_SIGN['level' if axis == 'level' else 'location']
                if name == 'S-P80':
                    block = CAMP[CAMP.time.eq(t) & CAMP.statistic.eq('level')].set_index('set')
                    members = [p for p in items if p in block.index]
                    k = int(replicated(block.z.reindex(members), block.fdr.reindex(members), pd.Series(PROGRAM_SIGN).reindex(members)).sum())
                    hits = set_hits(t, axis, signs, CAM)
                    pool = [s for s in S91.index if s in hits.index and signs.get(s, 0) != 0]
                    sizes_pool = CAM[CAM.time.eq(t) & CAM.statistic.eq('level')].set_index('set')['n in E']
                    sizes_list = block['n in E'].reindex(members)
                else:
                    hits = set_hits(t, axis, signs, cam)
                    pool = [s for s in universe if s in hits.index and (not signed or signs.get(s, 0) != 0)]
                    members = [s for s in items if s in set(pool)]
                    k = int(hits.reindex(members).fillna(False).sum())
                    sizes_pool = CAM[CAM.time.eq(t) & CAM.statistic.eq('level')].set_index('set')['n in E']
                    sizes_list = sizes_pool.reindex(members)
                cuts = np.quantile(sizes_pool.reindex(pool).to_numpy(float), [1 / 3, 2 / 3])
                strata_pool = np.digitize(sizes_pool.reindex(pool).to_numpy(float), cuts)
                strata_list = np.digitize(sizes_list.to_numpy(float), cuts)
                null = matched_null(hits.reindex(pool).fillna(False).to_numpy(), strata_pool, strata_list, n=N_NULL, seed=SEED)
            n = len(members)
            rows.append({'list': name, 'kind': kind, 'axis': axis, 'time': t, 'n': n, 'k': k,
                         'precision': k / n if n else np.nan, **null_summary(k, null),
                         'pool': len(pool), 'pool rate': float(hits.reindex(pool).fillna(False).mean()) if pool else np.nan})
SCORES = pd.DataFrame(rows)
SCORES['enrichment'] = SCORES.precision / (SCORES['null mean'] / SCORES.n)
SCORES['null precision high'] = SCORES['null high'] / SCORES.n
SCORES['replicates'] = (SCORES.n >= MIN_N_REPLICATE) & (SCORES.k > SCORES['null high'])
SCORES.to_csv(tables / 'list_scores.csv', index=False)
primary_view = SCORES[SCORES.apply(lambda r: PRIMARY[r['list']] == r['axis'], axis=1)]
display(primary_view.pivot_table(index=['list', 'axis'], columns='time', values='precision').reindex(columns=TIMES)
        .rename(columns=T_LABEL).round(3))
display(primary_view[primary_view.time.eq(T_STAR)][['list', 'axis', 'n', 'k', 'precision', 'null mean', 'null high',
                                                    'p', 'enrichment', 'replicates']].round(3))

### 7.1 · Comparisons and verdicts (rules 1–5)

In [ ]:
COMP = []
for axis in AXES:
    block = SCORES[SCORES.axis.eq(axis)]
    names = sorted(block['list'].unique())
    for a in names:
        for b in names:
            if a == b:
                continue
            row = {'axis': axis, 'A': a, 'B': b}
            beats = {}
            for t in TIMES:
                ra = block[block['list'].eq(a) & block.time.eq(t)].iloc[0]
                rb = block[block['list'].eq(b) & block.time.eq(t)].iloc[0]
                ok = ra.n >= MIN_N_COMPARE and rb.n >= MIN_N_COMPARE
                beats[t] = bool(ok and ra.precision - rb.precision >= MARGIN)
                row[f'Δ precision {T_LABEL[t]}'] = ra.precision - rb.precision if ok else np.nan
            others = [t for t in ACUTE if t != T_STAR]
            row['beats at t*'] = beats[T_STAR]
            row['stable'] = beats[T_STAR] and any(beats[t] for t in others)
            COMP.append(row)
COMP = pd.DataFrame(COMP)
COMP.to_csv(tables / 'comparisons.csv', index=False)

VERDICTS = []
for axis in AXES:
    block = SCORES[SCORES.axis.eq(axis) & SCORES.time.eq(T_STAR) & (SCORES.n >= MIN_N_COMPARE)]
    eligible = block['list'].tolist()
    best = [a for a in eligible if all(COMP[(COMP.axis == axis) & (COMP.A == a) & (COMP.B == b)].stable.iloc[0]
                                       for b in eligible if b != a)]
    VERDICTS.append({'axis': axis, 'lists with n ≥ 20': ', '.join(eligible), 'best': best[0] if best else 'none',
                     'replicating at t*': ', '.join(block['list'][block.replicates])})
star = SCORES[SCORES.time.eq(T_STAR)].set_index(['list', 'axis'])
f_sets = bool(star.loc[('S-F71', 'unsigned location'), 'replicates'])
f_genes = bool(star.loc[('G-F71', 'unsigned location'), 'replicates'])
F_VERDICT = 'replicates' if f_sets and f_genes else 'partly' if (f_sets or f_genes) else 'does not replicate'
VERDICTS = pd.DataFrame(VERDICTS)
display(VERDICTS)
print('Positional F-test, AKI shape calls (rule 4):', F_VERDICT, f'(sets {f_sets}, genes {f_genes})')

## 8 · Threshold-free agreement

Spearman ρ over all items between an internal statistic and the external statistic of its axis, per time point.

In [ ]:
f_z = pd.Series(norm.isf(cam71.p_one_sided.astype(float).clip(1e-300, 1 - 1e-16)), index=cam71.index)
AGREE = []
for t in TIMES:
    e = EXT[t]
    gl = [g for g in E if g in G91.index]
    gf = [g for g in E if g in stat71.index and usable71.get(g, False)]
    lvl = CAM[CAM.time.eq(t) & CAM.statistic.eq('level')].set_index('set').z
    loc = CAM[CAM.time.eq(t) & CAM.statistic.eq('location')].set_index('set').z
    aloc = CAM[CAM.time.eq(t) & CAM.statistic.eq('|location|')].set_index('set').z
    s91 = [s for s in S91.index if s in lvl.index]
    sf = [s for s in cam71.index if s in aloc.index]
    pairs = {
        'genes · L vs level': (G91.loc[gl, 'L'], e.loc[gl, 'level coefficient']),
        'genes · Δ late − L vs location': (G91.loc[gl, 'Δ late'] - G91.loc[gl, 'L'], e.loc[gl, 'location coefficient']),
        'genes · F z vs |location t|': (stat71.loc[gf, 'z'].astype(float), e.loc[gf, 'location t'].abs()),
        'sets · 91 z L vs level z': (S91.loc[s91, 'z L'], lvl[s91]),
        'sets · 91 z M × location sign vs location z': (S91.loc[s91, 'z M'] * SET_SIGN['location'][s91], loc[s91]),
        'sets · F z vs |location| z': (f_z[sf], aloc[sf])}
    for k, (a, b) in pairs.items():
        ok = np.isfinite(np.asarray(a, float)) & np.isfinite(np.asarray(b, float))
        AGREE.append({'pair': k, 'time': T_LABEL[t], 'items': int(ok.sum()),
                      'ρ': spearmanr(np.asarray(a, float)[ok], np.asarray(b, float)[ok])[0]})
AGREE = pd.DataFrame(AGREE)
AGREE.to_csv(tables / 'agreement.csv', index=False)
display(AGREE.pivot(index='pair', columns='time', values='ρ')[[T_LABEL[t] for t in TIMES]].round(3))

## 9 · Claim checks (named in the protocol)

In [ ]:
CLAIMS = []
for name, (members, axis, direction) in CLAIM_PROGRAMS.items():
    for t in TIMES:
        stat = 'level' if axis == 'level' else 'location'
        block = CAMP[CAMP.time.eq(t) & CAMP.statistic.eq(stat)].set_index('set')
        r = block.loc[name] if name in block.index else None
        CLAIMS.append({'check': name, 'axis': axis, 'expected': '+' if direction > 0 else '−', 'time': T_LABEL[t],
                       'members in E': int(r['n in E']) if r is not None else 0,
                       'z': float(r.z) if r is not None else np.nan, 'fdr': float(r.fdr) if r is not None else np.nan,
                       'holds': bool(r is not None and r.fdr <= ALPHA and np.sign(r.z) == direction)})
for group, ids, direction in (('K5', K5, -1.), ('K6', K6, 1.)):
    for s in ids:
        for t in TIMES:
            block = CAM[CAM.time.eq(t) & CAM.statistic.eq('location')].set_index('set')
            r = block.loc[s] if s in block.index else None
            CLAIMS.append({'check': f'{group} · {s.split("::", 1)[1]}', 'axis': 'location',
                           'expected': '+' if direction > 0 else '−', 'time': T_LABEL[t],
                           'members in E': int(r['n in E']) if r is not None else 0,
                           'z': float(r.z) if r is not None else np.nan, 'fdr': float(r.fdr) if r is not None else np.nan,
                           'holds': bool(r is not None and r.fdr <= ALPHA and np.sign(r.z) == direction)})
CLAIMS = pd.DataFrame(CLAIMS)
CLAIMS.to_csv(tables / 'claim_checks.csv', index=False)
display(CLAIMS[CLAIMS.time.eq(T_LABEL[T_STAR])].round(3))
display(CLAIMS.pivot(index='check', columns='time', values='holds')[[T_LABEL[t] for t in TIMES]])

## 10 · Figures

- 94a: time-point alignment.
- 94b: each list on its primary axis: precision against its matched null at t*, and enrichment over time points.
- 94c: claim checks.
- 94d: internal against external set statistics at t*.

In [ ]:
fig, ax = plt.subplots(figsize=(70 * MM, 45 * MM))
x = np.arange(len(TIMES))
for col, color in (('level ρ', INK), ('location ρ', '#9C93D3'), ('location (assigned) ρ', LIGHT)):
    ax.plot(x, ALIGN[col].to_numpy(), marker='o', ms=3, color=color, label=col.replace(' ρ', ''))
ax.axhline(0, color=MUTED, lw=.5)
ax.axvline(TIMES.index(T_STAR), color=MUTED, lw=.5, ls=':')
ax.set_xticks(x, [T_LABEL[t] for t in TIMES])
ax.set_ylabel('Spearman ρ, ours vs IRI')
ax.legend(loc='best')
save(fig, '94a_alignment')
plt.show()

order = ['G-L91', 'G-C71', 'S-L91', 'S-C71', 'S-P80', 'G-M91', 'S-M91', 'S-N93', 'G-F71', 'S-F71', 'S-F76']
fig, axes = plt.subplots(1, 2, figsize=(150 * MM, 62 * MM), gridspec_kw={'width_ratios': [1, 1.3]})
ax = axes[0]
pv = primary_view[primary_view.time.eq(T_STAR)].set_index('list').reindex(order)
y = np.arange(len(order))[::-1]
for yi, (name, r) in zip(y, pv.iterrows()):
    ax.plot([r['null low'] / r.n, r['null high'] / r.n], [yi, yi], color=LIGHT, lw=3, solid_capstyle='butt')
    ax.plot(r['null mean'] / r.n, yi, '|', color=MUTED, ms=6)
    ax.plot(r.precision, yi, 'o', color=LIST_COLOR[name], ms=4)
ax.set_yticks(y, [f'{n} · {PRIMARY[n]}' for n in order])
ax.set_xlabel(f'share replicated at {T_LABEL[T_STAR]}')
ax.set_xlim(0, 1)
ax = axes[1]
for name in order:
    r = primary_view[primary_view['list'].eq(name)].set_index('time').reindex(TIMES)
    ax.plot(x, r.enrichment.to_numpy(), marker='o', ms=2.5, color=LIST_COLOR[name],
            ls='-' if name.startswith('S-') else '--', label=name)
ax.axhline(1, color=MUTED, lw=.5)
ax.axvline(TIMES.index(T_STAR), color=MUTED, lw=.5, ls=':')
ax.set_xticks(x, [T_LABEL[t] for t in TIMES])
ax.set_ylabel('precision / matched-null precision')
ax.legend(loc='upper left', bbox_to_anchor=(1, 1), ncol=1)
fig.tight_layout()
save(fig, '94b_list_scores')
plt.show()

piv_z = CLAIMS.pivot(index='check', columns='time', values='z')[[T_LABEL[t] for t in TIMES]]
piv_h = CLAIMS.pivot(index='check', columns='time', values='holds')[[T_LABEL[t] for t in TIMES]]
fig, ax = plt.subplots(figsize=(80 * MM, 4 * MM * len(piv_z) + 15 * MM))
lim = np.nanpercentile(np.abs(piv_z.to_numpy()), 95)
im = ax.imshow(piv_z.to_numpy(), cmap='RdBu_r', vmin=-lim, vmax=lim, aspect='auto')
for (i, j), h in np.ndenumerate(piv_h.to_numpy()):
    if h:
        ax.plot(j, i, 'o', color=INK, ms=2.5)
ax.set_xticks(range(len(TIMES)), [T_LABEL[t] for t in TIMES])
ax.set_yticks(range(len(piv_z)), piv_z.index)
ax.spines[:].set_visible(False)
fig.colorbar(im, ax=ax, shrink=.6, label='external camera z')
save(fig, '94c_claim_checks')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(130 * MM, 58 * MM))
lvl = CAM[CAM.time.eq(T_STAR) & CAM.statistic.eq('level')].set_index('set').z
aloc = CAM[CAM.time.eq(T_STAR) & CAM.statistic.eq('|location|')].set_index('set').z
s91 = [s for s in S91.index if s in lvl.index]
called = S91.loc[s91, 'level set'].astype(bool)
ax = axes[0]
ax.scatter(S91.loc[s91, 'z L'][~called], lvl[s91][~called], s=3, color=LIGHT, lw=0)
ax.scatter(S91.loc[s91, 'z L'][called], lvl[s91][called], s=4, color=LIST_COLOR['S-L91'], lw=0)
ax.axhline(0, color=MUTED, lw=.5); ax.axvline(0, color=MUTED, lw=.5)
ax.set_xlabel('our set level z'); ax.set_ylabel(f'IRI level z ({T_LABEL[T_STAR]})')
sf = [s for s in cam71.index if s in aloc.index]
fc = cam71.loc[sf, 'call'].astype(bool)
ax = axes[1]
ax.scatter(f_z[sf][~fc], aloc[sf][~fc], s=3, color=LIGHT, lw=0)
ax.scatter(f_z[sf][fc], aloc[sf][fc], s=4, color=LIST_COLOR['S-F71'], lw=0)
ax.axhline(0, color=MUTED, lw=.5); ax.axvline(0, color=MUTED, lw=.5)
ax.set_xlabel('positional F-test set z'); ax.set_ylabel(f'IRI |location| z ({T_LABEL[T_STAR]})')
fig.tight_layout()
save(fig, '94d_set_agreement')
plt.show()

## 11 · How the protocol was read, and the run record

In [ ]:
READING = pd.DataFrame([
    ('amended before any fit (1599681)', "Set location sign over the bins centred ≥ 1.8 (91's LATE); the frozen text said 2.1."),
    ('amended before any fit (1599681)', "G-C71: clustering's selected genes with direction Up or Down (the protocol's 'up and down groups')."),
    ('amended before any fit (1599681)', 'Programs are tested on their members in E when ≥ 5 remain (library sets: 10–300).'),
    ('reading', 'Library set members in E are the union of the set across notebooks 71, 76 and 91 (one library), intersected with E.'),
    ('reading', 'Programs (80, claim checks) form their own camera family, so their BH does not change the library sets.'),
    ('reading', "S-P80's null pool is 91's library sets with their own level sign, matched on size tercile."),
    ('reading', "Gene expression strata: the percentile of 91's control log2 CP10k, else of clustering's expression per 10k."),
    ('reading', "K5 names Hallmark Fatty Acid Metabolism as frozen; 91's location set of that name was Reactome's (R-HSA-8978868)."),
    ('reading', 'A list is scored on every axis it can be (signed axes need a nonzero internal sign); its primary axis decides.'),
    ('reading', "Clustering's direction labels are lowercase (up, down, crossover); G-C71 reads up and down."),
    ('reading', "76's programs.csv truncates members at 60; C is rebuilt as Hallmark G2-M ∪ E2F in U76 (297) and T_PT from "
                "programs_transporters.csv (93), both checked against 76's counts."),
], columns=['kind', 'what'])
READING.to_csv(tables / 'how_the_protocol_was_read.csv', index=False)
display(READING)

RUN = {'notebook': '94_pt_external_iri', 'logic_version': NOTEBOOK_LOGIC_VERSION, 'seed': SEED,
       'guards': guard_log, 't_star': T_LABEL[T_STAR], 'universe E': len(E), 'library sets': len(LIB_SETS),
       'programs': len(PROG_SETS), 'assignment accuracy (mean)': float(ACC.accuracy.mean()),
       'F-test verdict (rule 4)': F_VERDICT, 'best per axis': dict(zip(VERDICTS.axis, VERDICTS.best)),
       'seconds': round(time.perf_counter() - NOTEBOOK_START, 1),
       'versions': {m: __import__(m).__version__ for m in ('numpy', 'pandas', 'scipy', 'matplotlib')}}
(output / 'run_record.json').write_text(json.dumps(RUN, indent=2))
print(json.dumps(RUN, indent=2))